In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import time

CATALOG = "fahrlytics"
SILVER  = f"{CATALOG}.silver"
GOLD    = f"{CATALOG}.gold"

spark.sql(f"USE CATALOG {CATALOG}")

print(f"Catalog           : {CATALOG}")
print(f"Source Tables     : {SILVER}.clickstream, {SILVER}.trips")
print(f"Target Gold Table : {GOLD}.funnel_metrics")

In [0]:
print("Reading clickstream events and projecting session milestones...")

df_clickstream = spark.table(f"{SILVER}.clickstream")

# Roll up session-level flags prior to dimensional slicing
session_level_summary = (
    df_clickstream.groupBy(
        "session_id",
        "customer_id",
        "event_date",
        "city",
        "device_type",
        "ab_test_group",
        "ab_test_experiment"
    )
    .agg(
        F.max("is_app_open").alias("has_opened_app"),
        F.max("is_search").alias("has_searched"),
        F.max("is_quote_view").alias("has_viewed_quote"),
        F.max("is_booking_confirmed").alias("has_confirmed_booking"),
        F.max("is_booking_cancelled").alias("has_cancelled_booking"),
        F.max("is_bounce_session").alias("has_bounced"),
        F.max("session_duration_sec").alias("session_duration_sec"),
        F.max("total_session_events").alias("session_event_count")
    )
)

In [0]:
print("Aggregating metrics by date, city, device, and A/B variant...")

funnel_aggregates = (
    session_level_summary.groupBy(
        "event_date",
        "city",
        "device_type",
        "ab_test_group",
        "ab_test_experiment"
    )
    .agg(
        # Top-line traffic
        F.countDistinct("session_id").alias("total_sessions"),
        F.countDistinct("customer_id").alias("unique_active_users"),

        # Funnel stage counts
        F.sum(F.when(F.col("has_opened_app") == True, 1).otherwise(0)).alias("funnel_step1_app_opens"),
        F.sum(F.when(F.col("has_searched") == True, 1).otherwise(0)).alias("funnel_step2_searches"),
        F.sum(F.when(F.col("has_viewed_quote") == True, 1).otherwise(0)).alias("funnel_step3_quote_views"),
        F.sum(F.when(F.col("has_confirmed_booking") == True, 1).otherwise(0)).alias("funnel_step4_confirmed_bookings"),
        F.sum(F.when(F.col("has_cancelled_booking") == True, 1).otherwise(0)).alias("cancelled_bookings"),
        F.sum(F.when(F.col("has_bounced") == True, 1).otherwise(0)).alias("bounced_sessions"),

        # Latencies
        F.round(F.avg("session_duration_sec"), 1).alias("avg_session_duration_sec"),
        F.round(F.avg("session_event_count"), 1).alias("avg_events_per_session")
    )
)

In [0]:
print("Calculating step-by-step conversion ratios...")

df_funnel_ratios = (
    funnel_aggregates
    # Step conversion rates
    .withColumn("search_rate",
        F.round(F.col("funnel_step2_searches") / F.greatest(F.col("funnel_step1_app_opens"), F.lit(1)), 4)
    )
    .withColumn("quote_view_rate",
        F.round(F.col("funnel_step3_quote_views") / F.greatest(F.col("funnel_step2_searches"), F.lit(1)), 4)
    )
    .withColumn("quote_to_booking_rate",
        F.round(F.col("funnel_step4_confirmed_bookings") / F.greatest(F.col("funnel_step3_quote_views"), F.lit(1)), 4)
    )
    # End-to-end session conversion rate: open -> confirmed booking
    .withColumn("overall_conversion_rate",
        F.round(F.col("funnel_step4_confirmed_bookings") / F.greatest(F.col("total_sessions"), F.lit(1)), 4)
    )
    # Session bounce & cancellation rates
    .withColumn("bounce_rate",
        F.round(F.col("bounced_sessions") / F.greatest(F.col("total_sessions"), F.lit(1)), 4)
    )
    .withColumn("in_app_cancellation_rate",
        F.round(F.col("cancelled_bookings") / F.greatest(F.col("total_sessions"), F.lit(1)), 4)
    )
    .withColumn("_calculated_at", F.current_timestamp())
)

In [0]:
print(f"Writing datamart to {GOLD}.funnel_metrics...")
t0 = time.time()

(
    df_funnel_ratios.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("event_date")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.funnel_metrics")
)

print(f"Gold funnel metrics written in {round(time.time() - t0, 2)}s.")

print("Running OPTIMIZE with Z-ORDER on city and ab_test_group...")
spark.sql(f"OPTIMIZE {GOLD}.funnel_metrics ZORDER BY (city, ab_test_group)")
print("OPTIMIZE completed.")

In [0]:
print("=== FAHRLYTICS A/B EXPERIMENT & FUNNEL AUDIT ===\n")

df_gold_funnel = spark.table(f"{GOLD}.funnel_metrics")
total_rows = df_gold_funnel.count()
print(f"Total dimensional slices: {total_rows:,}\n")

print("A/B Variant Conversion Comparison (Lifetime Aggregate):")
df_ab_summary = (
    df_gold_funnel.groupBy("ab_test_group")
    .agg(
        F.sum("total_sessions").alias("total_sessions"),
        F.sum("unique_active_users").alias("unique_users"),
        F.sum("funnel_step4_confirmed_bookings").alias("total_bookings"),
        F.round(
            F.sum("funnel_step4_confirmed_bookings") / F.sum("total_sessions"), 4
        ).alias("overall_conversion_rate"),
        F.round(
            F.sum("bounced_sessions") / F.sum("total_sessions"), 4
        ).alias("bounce_rate"),
        F.round(F.avg("avg_session_duration_sec"), 1).alias("avg_session_duration_sec")
    )
)
df_ab_summary.show(truncate=False)

# Compute Observed Variant Lift Percentage
summary_rows = {row["ab_test_group"]: row["overall_conversion_rate"] for row in df_ab_summary.collect()}
if "control" in summary_rows and "variant" in summary_rows:
    ctrl_cr = summary_rows["control"]
    var_cr  = summary_rows["variant"]
    lift    = ((var_cr - ctrl_cr) / ctrl_cr) * 100.0
    print(f"Observed Conversion Lift for Variant: {lift:+.2f}%\n")

print("Conversion Breakdown by City:")
df_gold_funnel.groupBy("city").agg(
    F.sum("total_sessions").alias("total_sessions"),
    F.sum("funnel_step4_confirmed_bookings").alias("confirmed_bookings"),
    F.round(
        F.sum("funnel_step4_confirmed_bookings") / F.sum("total_sessions"), 4
    ).alias("city_conversion_rate")
).orderBy(F.col("total_sessions").desc()).show(truncate=False)

print("Device Breakdown:")
df_gold_funnel.groupBy("device_type").agg(
    F.sum("total_sessions").alias("total_sessions"),
    F.round(
        F.sum("funnel_step4_confirmed_bookings") / F.sum("total_sessions"), 4
    ).alias("device_conversion_rate"),
    F.round(
        F.sum("bounced_sessions") / F.sum("total_sessions"), 4
    ).alias("bounce_rate")
).show()